# CS6603 Final Project — Group 109
## Step 4: Mitigating Bias

This notebook picks up from Step 3, where a pre-processing bias mitigation
algorithm (**Reweighing**) was used to create a *transformed* version of the
`student-por.csv` dataset. Step 4 checks whether that mitigation actually
helped, by:

1. Training a classifier on the **original** dataset and measuring fairness
   metrics on a held-out test set.
2. Training a classifier on the **transformed** (reweighed) dataset and
   measuring the same fairness metrics on the same test structure.
3. Comparing the two sets of results.

Protected classes used throughout: **sex** and **age** (as selected in Step 2.3).
Dependent variable used to train the classifier: **G3** (final grade),
binarized into **pass (>=10)** / **fail (<10)**, consistent with Step 1.5.

> NOTE: Adjust `PRIVILEGED_SEX`, `AGE_THRESHOLD`, and `TARGET_COL` below if your
> Step 3 report defined the privileged/unprivileged groups differently — the
> rest of the notebook will follow automatically.


## 0. Imports

In [ ]:
import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load the Dataset

The dataset is the UCI **Student Performance** dataset. Since the report
documents 649 observations, we use `student-por.csv` (the Portuguese-language
course file), loaded with `;` as the separator, matching Step 1.3.


In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

def load_student_data(url=DATA_URL, fname="student-por.csv"):
    """Download and load the student performance dataset.
    Falls back to a local copy at ./student-por.csv if offline."""
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            zdata = resp.read()
        with zipfile.ZipFile(io.BytesIO(zdata)) as z:
            with z.open(fname) as f:
                df = pd.read_csv(f, sep=';')
    except Exception as e:
        print(f"Download failed ({e}); trying local file '{fname}'...")
        df = pd.read_csv(fname, sep=';')
    return df

df = load_student_data()
print(df.shape)
df.head()


## 2. Define Protected Classes, Privileged/Unprivileged Groups, and the Target

- **Protected class 1 — sex**: privileged group vs. unprivileged group.
- **Protected class 2 — age**: privileged group vs. unprivileged group, split
  at `AGE_THRESHOLD`.
- **Target (G3)**: binarized to `1 = pass` (G3 >= 10), `0 = fail`, per common
  grading conventions for this dataset (0-20 scale, 10 = passing).


In [ ]:
# ---- configure privileged / unprivileged groups here (align with Step 3) ----
PRIVILEGED_SEX_LABEL = "M"      # privileged group for 'sex'
UNPRIVILEGED_SEX_LABEL = "F"    # unprivileged group for 'sex'
AGE_THRESHOLD = 18              # age <= threshold -> privileged, > threshold -> unprivileged
TARGET_COL = "G3"
PASS_THRESHOLD = 10
# -----------------------------------------------------------------------------

data = df.copy()

# Encode sex per Table 2 in the report: F=0, M=1
data['sex_code'] = data['sex'].map({'F': 0, 'M': 1})
data['sex_privileged'] = (data['sex'] == PRIVILEGED_SEX_LABEL).astype(int)

# Age privileged/unprivileged flag
data['age_privileged'] = (data['age'] <= AGE_THRESHOLD).astype(int)

# Binarize the outcome variable G3 -> pass/fail
data['label'] = (data[TARGET_COL] >= PASS_THRESHOLD).astype(int)

print(data['sex'].value_counts())
print(data['age_privileged'].value_counts())
print(data['label'].value_counts())


## 3. Fairness Metrics (same two selected in Step 3.2)

We use the two fairness metrics already selected in Step 3.2:

- **Statistical Parity Difference (SPD)** =
  P(Ŷ=1 | unprivileged) − P(Ŷ=1 | privileged)
- **Disparate Impact (DI)** =
  P(Ŷ=1 | unprivileged) / P(Ŷ=1 | privileged)

Both are computed on model **predictions** (Ŷ), for each protected class,
on the held-out test set.


In [ ]:
def statistical_parity_difference(y_pred, privileged_mask):
    priv_rate = y_pred[privileged_mask].mean() if privileged_mask.sum() > 0 else np.nan
    unpriv_rate = y_pred[~privileged_mask].mean() if (~privileged_mask).sum() > 0 else np.nan
    return unpriv_rate - priv_rate

def disparate_impact(y_pred, privileged_mask):
    priv_rate = y_pred[privileged_mask].mean() if privileged_mask.sum() > 0 else np.nan
    unpriv_rate = y_pred[~privileged_mask].mean() if (~privileged_mask).sum() > 0 else np.nan
    if priv_rate == 0:
        return np.nan
    return unpriv_rate / priv_rate

def compute_fairness_metrics(y_pred, privileged_mask):
    return {
        "Statistical Parity Difference": statistical_parity_difference(y_pred, privileged_mask),
        "Disparate Impact": disparate_impact(y_pred, privileged_mask),
    }


## 4. Reweighing (Pre-Processing Bias Mitigation, from Step 3.3)

We recreate the transformed dataset from Step 3.3 using the classic
**Reweighing** algorithm (Kamiran & Calders, 2012): each instance is assigned
a weight so that, in expectation, the protected attribute and the label
become statistically independent:

`weight(a, y) = [P(A=a) * P(Y=y)] / P(A=a, Y=y)`

This does not change any feature values (so it can be applied identically to
both protected classes); it only changes the *sample weights* used when
fitting the classifier — this is the standard AIF360 `Reweighing`
pre-processing approach.


In [ ]:
def reweighing_weights(protected_attr, label):
    """Return per-instance weights that balance protected_attr against label."""
    protected_attr = np.asarray(protected_attr)
    label = np.asarray(label)
    n = len(label)
    weights = np.ones(n, dtype=float)

    for a in np.unique(protected_attr):
        p_a = (protected_attr == a).mean()
        for y in np.unique(label):
            p_y = (label == y).mean()
            mask = (protected_attr == a) & (label == y)
            p_ay = mask.mean()
            if p_ay > 0:
                weights[mask] = (p_a * p_y) / p_ay
    return weights

# Combine sex + age privilege status into one joint protected-attribute weighting
# so both protected classes selected in Step 2.3 are addressed simultaneously.
joint_group = data['sex_privileged'].astype(str) + "_" + data['age_privileged'].astype(str)
data['instance_weight'] = reweighing_weights(joint_group, data['label'])

data[['sex', 'age', 'sex_privileged', 'age_privileged', 'label', 'instance_weight']].head(10)


## 5. Build the Feature Matrix

Features: all columns except the three grade columns (`G1`, `G2`, `G3`,
since those are the dependent variables) and the helper columns created
above. Categorical columns are one-hot encoded.


In [ ]:
drop_cols = ['G1', 'G2', 'G3', 'label', 'sex_privileged', 'age_privileged', 'instance_weight', 'sex_code']
feature_df = data.drop(columns=drop_cols)

categorical_cols = feature_df.select_dtypes(include='object').columns.tolist()
X = pd.get_dummies(feature_df, columns=categorical_cols, drop_first=True)
y = data['label'].values

sex_privileged = data['sex_privileged'].values.astype(bool)
age_privileged = data['age_privileged'].values.astype(bool)
weights = data['instance_weight'].values

print(X.shape)


## 6. Step 4 — Original Dataset

1. Split the **original** dataset into train/test.
2. Train a classifier (Logistic Regression) using `G3` (binarized) as the label.
3. Compute the two fairness metrics on the test set for sex and age.


In [ ]:
# Single split, reused (with the same indices) for both the original and
# transformed-dataset experiments, so the test set is directly comparable.
indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y[idx_train], y[idx_test]
sex_priv_test = sex_privileged[idx_test]
age_priv_test = age_privileged[idx_test]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Train classifier on the ORIGINAL (unweighted) training data ---
clf_original = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
clf_original.fit(X_train_scaled, y_train)
y_pred_original = clf_original.predict(X_test_scaled)

acc_original = accuracy_score(y_test, y_pred_original)
print(f"Original-dataset classifier test accuracy: {acc_original:.3f}")

metrics_original_sex = compute_fairness_metrics(y_pred_original, sex_priv_test)
metrics_original_age = compute_fairness_metrics(y_pred_original, age_priv_test)

print("\nOriginal dataset — Sex fairness metrics:", metrics_original_sex)
print("Original dataset — Age fairness metrics:", metrics_original_age)


## 7. Step 4 — Transformed Dataset (Reweighed)

1. Split the **transformed** dataset (same train/test indices as above, so
   results are comparable) — the "transformation" here is the per-instance
   weight computed in Section 4.
2. Train a classifier on the transformed **training** data using the
   instance weights (`sample_weight`) from the Reweighing algorithm.
3. Compute the same fairness metrics on the test set.


In [ ]:
w_train = weights[idx_train]

clf_transformed = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
clf_transformed.fit(X_train_scaled, y_train, sample_weight=w_train)
y_pred_transformed = clf_transformed.predict(X_test_scaled)

acc_transformed = accuracy_score(y_test, y_pred_transformed)
print(f"Transformed-dataset (reweighed) classifier test accuracy: {acc_transformed:.3f}")

metrics_transformed_sex = compute_fairness_metrics(y_pred_transformed, sex_priv_test)
metrics_transformed_age = compute_fairness_metrics(y_pred_transformed, age_priv_test)

print("\nTransformed dataset — Sex fairness metrics:", metrics_transformed_sex)
print("Transformed dataset — Age fairness metrics:", metrics_transformed_age)


## 8. Privileged vs. Unprivileged — Difference Table

For each fairness metric, indicate whether there was a difference in
outcomes between the privileged and unprivileged groups, for both the
original and transformed classifiers.


In [ ]:
def priv_unpriv_rates(y_pred, privileged_mask):
    priv_rate = y_pred[privileged_mask].mean()
    unpriv_rate = y_pred[~privileged_mask].mean()
    return priv_rate, unpriv_rate

rows = []
for dataset_name, y_pred in [("Original", y_pred_original), ("Transformed", y_pred_transformed)]:
    for pc_name, mask in [("Sex", sex_priv_test), ("Age", age_priv_test)]:
        priv_rate, unpriv_rate = priv_unpriv_rates(y_pred, mask)
        difference = "Yes" if not np.isclose(priv_rate, unpriv_rate, atol=1e-6) else "No"
        rows.append({
            "Dataset": dataset_name,
            "Protected Class": pc_name,
            "Privileged Positive Rate": round(priv_rate, 4),
            "Unprivileged Positive Rate": round(unpriv_rate, 4),
            "Difference Between Groups?": difference,
        })

priv_unpriv_table = pd.DataFrame(rows)
priv_unpriv_table


## 9. Positive / Negative / No Change Table

Two comparisons, per fairness metric and per protected class:

- **Effect of transforming the dataset**: comparing the fairness metric
  computed on the transformed classifier vs. the original classifier.
- **Effect of training the classifier**: comparing the fairness metric on the
  transformed testing dataset vs. the original testing dataset (same
  comparison, reported per the assignment's requested framing).

For **Statistical Parity Difference**, a value closer to 0 is fairer, so a
change is "positive" if `|SPD|` decreased.
For **Disparate Impact**, a value closer to 1 is fairer, so a change is
"positive" if `|DI - 1|` decreased.


In [ ]:
def classify_change(old_value, new_value, metric_name):
    if metric_name == "Statistical Parity Difference":
        old_dist, new_dist = abs(old_value), abs(new_value)
    else:  # Disparate Impact
        old_dist, new_dist = abs(old_value - 1), abs(new_value - 1)

    if np.isclose(old_dist, new_dist, atol=1e-6):
        return "No change"
    return "Positive change" if new_dist < old_dist else "Negative change"

change_rows = []
metric_names = ["Statistical Parity Difference", "Disparate Impact"]
protected_classes = [("Sex", metrics_original_sex, metrics_transformed_sex),
                     ("Age", metrics_original_age, metrics_transformed_age)]

for pc_name, orig_metrics, trans_metrics in protected_classes:
    for metric_name in metric_names:
        old_value = orig_metrics[metric_name]
        new_value = trans_metrics[metric_name]
        change_rows.append({
            "Protected Class": pc_name,
            "Fairness Metric": metric_name,
            "Original Dataset Value": round(old_value, 4),
            "Transformed Dataset Value": round(new_value, 4),
            "Change After Transforming Dataset": classify_change(old_value, new_value, metric_name),
        })

change_table = pd.DataFrame(change_rows)
change_table


## 10. Summary of All 8 Fairness Metric Values

Matches the table format requested in Step 3.2 / Step 3.4 (2 fairness
metrics × 2 protected classes × original/transformed = 8 values each phase).


In [ ]:
summary_rows = []
for dataset_name, sex_m, age_m in [("Original", metrics_original_sex, metrics_original_age),
                                    ("Transformed", metrics_transformed_sex, metrics_transformed_age)]:
    for pc_name, m in [("Sex", sex_m), ("Age", age_m)]:
        for metric_name, value in m.items():
            summary_rows.append({
                "Dataset": dataset_name,
                "Protected Class": pc_name,
                "Fairness Metric": metric_name,
                "Value": round(value, 4),
            })

summary_table = pd.DataFrame(summary_rows)
summary_table


## 11. Visualization: Fairness Metrics Before vs. After Mitigation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pc_name, orig_m, trans_m in zip(
    axes,
    ["Sex", "Age"],
    [metrics_original_sex, metrics_original_age],
    [metrics_transformed_sex, metrics_transformed_age],
):
    labels = list(orig_m.keys())
    original_vals = [orig_m[k] for k in labels]
    transformed_vals = [trans_m[k] for k in labels]

    x = np.arange(len(labels))
    width = 0.35
    ax.bar(x - width/2, original_vals, width, label='Original')
    ax.bar(x + width/2, transformed_vals, width, label='Transformed (Reweighed)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15)
    ax.set_title(f"Fairness Metrics — {pc_name}")
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.legend()

plt.tight_layout()
plt.savefig("step4_fairness_comparison.png", dpi=150)
plt.show()


## 12. Notes for the Report

- `acc_original` and `acc_transformed` give the classifier accuracy before
  and after mitigation — useful to discuss the fairness/accuracy trade-off.
- `priv_unpriv_table` answers: *"For each fairness metric, was there a
  difference in outcomes for privileged vs. unprivileged groups?"*
- `change_table` answers: *"Was there a positive, negative, or no change in
  each fairness metric after transforming the dataset / after training the
  classifier?"*
- `summary_table` is the 8-value results table (2 metrics × 2 protected
  classes) for both the original and transformed datasets.
- Copy these three tables directly into the final report, and export
  `step4_fairness_comparison.png` as the accompanying chart.
